# Unified Pool Analytics Panel
This notebook provides a unified plotting interface for single- and multi-pool panels.

Spec examples:
- ['vp', ['xcp', 'xcp_half']] → one subplot overlaying all pools for virtual price, plus per‑pool subplots for xcp & xcp_half.
- With combine toggle on: grouped metrics are combined on a single subplot (all pools × metrics).


In [ ]:
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Union, Optional

# Metric aliases (short names)
metric_key_aliases: Dict[str, str] = {}

# Preferred colors and aliases
default_pool_colors: Dict[str, str] = {
    "yb_wBTC": "orange",
    "yb_cbBTC": "tab:blue",
    "yb_tBTC": "tab:green",
}
default_pool_aliases: Dict[str, str] = {
    "yb_wBTC": "wBTC",
    "yb_cbBTC": "cbBTC",
    "yb_tBTC": "tBTC",
}


def _resolve_metric_key(name: str) -> str:
    return metric_key_aliases.get(name, name)


def load_pools_data(
    pools: Dict[str, str],
    decimals: Dict[str, int],
    ts_null: Optional[float] = None,
    data_dir: str = "data_events",
) -> Dict[str, Dict[str, np.ndarray]]:
    pools_data: Dict[str, Dict[str, np.ndarray]] = {}
    for pool_key, address in pools.items():
        filename = f"{data_dir}/{address}.csv"
        df = pd.read_csv(filename)
        if ts_null is not None:
            idx = np.where(df["timestamp"].astype(float).values > ts_null)[0]
            if len(idx):
                df = df.iloc[idx[0] :]
        ts = df["timestamp"].astype(float).to_numpy()
        ts_dt = pd.to_datetime(ts, unit="s")
        blocks = df["block"].astype(float).to_numpy()
        virtual_price = df["virtual_price"].astype(float).to_numpy() / 1e18
        xcp_profit = df["xcp_profit"].astype(float).to_numpy() / 1e18
        price_oracle = df["price_oracle"].astype(float).to_numpy() / 1e18
        price_scale = df["price_scale"].astype(float).to_numpy() / 1e18
        donation_shares = df["donation_shares"].astype(float).to_numpy() / 1e18
        last_donation_release_ts = df["last_donation_release_ts"].astype(float).to_numpy()
        donation_protection_expiry_ts = df["donation_protection_expiry_ts"].astype(float).to_numpy()
        total_supply = df["totalSupply"].astype(float).to_numpy() / 1e18
        D = df["D"].astype(float).to_numpy() / 1e18
        balances_0 = df["balances_0"].astype(float).to_numpy() / 1e18
        balances_1 = df["balances_1"].astype(float).to_numpy() / (10 ** decimals[pool_key])
        spot_price_in = 1 / (
            df["spot_price_in"].astype(float).to_numpy() / (10 ** decimals[pool_key])
        )
        spot_price_out = df["spot_price_out"].astype(float).to_numpy() / 10**18 / 1e-5
        lp_price = df["lp_price"].astype(float).to_numpy() / 1e18
        lp_price_scale = 2 * virtual_price * np.sqrt(price_scale)
        lp_price_2 = D / (total_supply)
        user_supply = total_supply - donation_shares

        xcp_profit_half = (xcp_profit - 1) / 2 + 1
        donation_duration = 7 * 86_400
        protection_period = 60
        protection_factor = np.clip((donation_protection_expiry_ts - ts) / protection_period, 0, 1)
        t_elapsed = ts - last_donation_release_ts
        unlocked_shares = np.clip(
            donation_shares * t_elapsed / donation_duration, 0, donation_shares
        )
        vp_xcp_half_gap = virtual_price - xcp_profit_half
        available_shares = unlocked_shares * (1 - protection_factor)
        donations_proportion = np.divide(
            donation_shares,
            total_supply,
            out=np.zeros_like(donation_shares),
            where=total_supply != 0,
        )
        unlocked_proportion = np.divide(
            unlocked_shares,
            total_supply,
            out=np.zeros_like(unlocked_shares),
            where=total_supply != 0,
        )
        value_oracle = balances_0 + balances_1 * price_oracle
        value_pscale = balances_0 + balances_1 * price_scale
        abs_val_dif = value_pscale - value_oracle
        rel_val_dif = abs_val_dif / value_oracle
        virtual_price_growth = virtual_price - virtual_price[0]
        xcp_profit_growth = xcp_profit - xcp_profit[0]
        xcp_profit_half_growth = xcp_profit_half - xcp_profit_half[0]
        spot_mid = (spot_price_in + spot_price_out) / 2.0
        value_spot_mid = balances_0 + balances_1 * spot_mid
        denom = balances_1 * spot_mid
        crvusd_share = np.divide(
            balances_0, value_spot_mid, out=np.zeros_like(balances_0), where=denom != 0
        )
        donations_value = donation_shares * lp_price
        available_value = available_shares * lp_price
        ps_oracle_diff = np.abs(price_scale - price_oracle)
        pools_data[pool_key] = {
            "ts": ts,
            "ts_dt": ts_dt,
            "blocks": blocks,
            "virtual_price": virtual_price,
            "xcp_profit": xcp_profit,
            "price_oracle": price_oracle,
            "price_scale": price_scale,
            "donation_shares": donation_shares,
            "last_donation_release_ts": last_donation_release_ts,
            "donation_protection_expiry_ts": donation_protection_expiry_ts,
            "total_supply": total_supply,
            "D": D,
            "balances_0": balances_0,
            "balances_1": balances_1,
            "spot_price_in": spot_price_in,
            "spot_price_out": spot_price_out,
            "user_supply": user_supply,
            "xcp_profit_half": xcp_profit_half,
            "protection_factor": protection_factor,
            "t_elapsed": t_elapsed,
            "unlocked_shares": unlocked_shares,
            "available_shares": available_shares,
            "donations_proportion": donations_proportion,
            "unlocked_proportion": unlocked_proportion,
            "value_oracle": value_oracle,
            "value_pscale": value_pscale,
            "pscale_minus_oracle": abs_val_dif,
            "rel_pscale_minus_oracle": rel_val_dif,
            "virtual_price_growth": virtual_price_growth,
            "xcp_profit_growth": xcp_profit_growth,
            "xcp_profit_half_growth": xcp_profit_half_growth,
            "crvusd_share": crvusd_share,
            "lp_price": lp_price,
            "lp_price_scale": lp_price_scale,
            "D/TS": lp_price_2,
            "vp_xcp_half_gap": vp_xcp_half_gap,
            "donations_value": donations_value,
            "available_value": available_value,
            "ps_oracle_diff": ps_oracle_diff,
            "spot_mid": spot_mid,
            "value_spot_mid": value_spot_mid,
        }
    return pools_data


def _slice_by_time(ts: np.ndarray, t_start: Optional[float], t_stop: Optional[float]) -> slice:
    if t_start is None and t_stop is None:
        return slice(None)
    start_idx = 0
    stop_idx = len(ts)
    if t_start is not None:
        idxs = np.where(ts > t_start)[0]
        if len(idxs):
            start_idx = int(idxs[0])
    if t_stop is not None:
        idxs = np.where(ts < t_stop)[0]
        if len(idxs):
            stop_idx = int(idxs[-1]) + 1
    return slice(start_idx, stop_idx)


def plot_unified_panel(
    pools_selection: Dict[str, str],
    pools_data: Dict[str, Dict[str, np.ndarray]],
    spec: List[Union[str, List[str]]],
    config: Optional[Dict] = None,
    labels: Optional[Dict[str, str]] = None,
    pool_colors: Optional[Dict[str, str]] = None,
    pool_aliases: Optional[Dict[str, str]] = None,
    t_start: Optional[float] = None,
    t_stop: Optional[float] = None,
):
    cfg = config or {}
    combine_group = bool(cfg.get("combine_grouped_on_one", False))
    sharex = bool(cfg.get("sharex", True))
    sharey = bool(cfg.get("sharey", False))
    ncols = cfg.get("ncols")
    nrows_cfg = cfg.get("nrows")
    width_per_col, height_per_row = cfg.get("figsize_per", (6, 3))

    pool_colors = pool_colors or default_pool_colors
    pool_aliases = pool_aliases or default_pool_aliases
    pools_keys = list(pools_selection.keys())

    # Build units list
    units: List[Tuple[str, Union[str, List[str]], Optional[str]]] = []

    def normalize_item(it):
        if isinstance(it, str):
            return "single", _resolve_metric_key(it), None
        elif isinstance(it, (list, tuple)):
            metrics = [_resolve_metric_key(m) for m in it]
            return "group", metrics, None
        else:
            raise ValueError(f"Unsupported spec item: {it}")

    norm = [normalize_item(it) for it in spec]
    for kind, payload, _ in norm:
        if kind == "single":
            units.append((kind, payload, None))
        else:
            metrics = payload
            if combine_group:
                units.append(("group", metrics, None))
            else:
                for pk in pools_keys:
                    units.append(("group", metrics, pk))

    nplots = len(units)
    if nplots == 0:
        raise ValueError("Empty spec produced no plots")
    if nrows_cfg is not None and ncols is not None:
        nrows = int(nrows_cfg)
    else:
        if ncols is None:
            ncols = min(3, max(1, int(math.ceil(math.sqrt(nplots)))))
        nrows = int(math.ceil(nplots / ncols)) if nrows_cfg is None else int(nrows_cfg)

    figsize = (width_per_col * ncols, height_per_row * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=sharex, sharey=sharey)
    axes_flat = axes.flatten() if isinstance(axes, np.ndarray) else np.array([axes])

    # Metric color mapping for grouped plots (solid lines only)
    metric_color: Dict[str, str] = {}
    palette = plt.rcParams.get("axes.prop_cycle", None)
    if palette is not None:
        base_colors = [c["color"] for c in palette]
    else:
        base_colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "tab:brown"]

    def ensure_metric_color(m: str):
        if m not in metric_color:
            metric_color[m] = base_colors[len(metric_color) % len(base_colors)]

    lbl = labels or {}

    def metric_label(m: str) -> str:
        return lbl.get(m, m)

    idx_ax = 0
    for unit in units:
        ax = axes_flat[idx_ax]
        kind, payload, pk = unit
        if kind == "single":
            metric = payload
            for pool_key in pools_keys:
                data = pools_data.get(pool_key, {})
                if metric not in data:
                    continue
                ts = data["ts"].astype(float)
                sl = _slice_by_time(ts, t_start, t_stop)
                ax.plot(
                    data["ts_dt"][sl],
                    data[metric][sl],
                    label=pool_aliases.get(pool_key, pool_key),
                    color=pool_colors.get(pool_key),
                )
            ax.set_title(metric_label(metric))
            ax.legend()
            ax.grid(alpha=0.2)
        else:
            metrics = payload
            for m in metrics:
                ensure_metric_color(m)
            if pk is None and combine_group:
                for pool_key in pools_keys:
                    data = pools_data.get(pool_key, {})
                    ts = data.get("ts")
                    if ts is None:
                        continue
                    ts = ts.astype(float)
                    sl = _slice_by_time(ts, t_start, t_stop)
                    for i_m, m in enumerate(metrics):
                        if m not in data:
                            continue
                        lw_seq = [2.2, 1.6, 1.2, 1.0]
                        alpha_seq = [1.0, 0.8, 0.6, 0.5]
                        ax.plot(
                            data["ts_dt"][sl],
                            data[m][sl],
                            label=f"{pool_aliases.get(pool_key, pool_key)}: {metric_label(m)}",
                            color=pool_colors.get(pool_key),
                            linestyle="-",
                            linewidth=lw_seq[i_m % len(lw_seq)],
                            alpha=alpha_seq[i_m % len(alpha_seq)],
                        )
                ax.set_title("; ".join(metric_label(m) for m in metrics))
                ax.legend()
                ax.grid(alpha=0.2)
            else:
                assert pk is not None
                data = pools_data.get(pk, {})
                ts = data.get("ts")
                sl = slice(None)
                if ts is not None:
                    ts = ts.astype(float)
                    sl = _slice_by_time(ts, t_start, t_stop)
                for m in metrics:
                    if m not in data:
                        continue
                    ax.plot(
                        data["ts_dt"][sl],
                        data[m][sl],
                        label=metric_label(m),
                        color=metric_color[m],
                        linestyle="-",
                    )
                ax.set_title(f"{pk}")
                ax.legend()
                ax.grid(alpha=0.2)
        idx_ax += 1

    for j in range(idx_ax, len(axes_flat)):
        axes_flat[j].axis("off")
    for a in axes_flat:
        for tick in a.get_xticklabels():
            tick.set_rotation(30)
            tick.set_ha("right")
    fig.tight_layout()
    return fig, axes

In [ ]:
# Pools and display aliases
pools = {
    "yb_cbBTC": "0x83f24023d15d835a213df24fd309c47dAb5BEb32",
    "yb_wBTC": "0xD9FF8396554A0d18B2CFbeC53e1979b7ecCe8373",
    "yb_tBTC": "0xf1F435B05D255a5dBdE37333C0f61DA6F69c6127",
}
decimals = {
    "yb_cbBTC": 8,
    "yb_wBTC": 8,
    "yb_tBTC": 18,
}

# Preferred colors (as requested)
pool_colors = {
    "yb_wBTC": "orange",
    "yb_cbBTC": "tab:blue",
    "yb_tBTC": "tab:green",
}
pool_aliases = {
    "yb_wBTC": "wBTC",
    "yb_cbBTC": "cbBTC",
    "yb_tBTC": "tBTC",
}

# Ensure recent data: run fetch if any pool is missing/stale
from pathlib import Path
import sys
import subprocess

MIN_OLD = 5  # minutes
DATA_DIRS = [Path("data_events"), Path("scripts/study/analyze_yb/data_events")]
CANDIDATE_SCRIPTS = [
    Path("fetch_data_events.py"),
    Path("scripts/study/analyze_yb/fetch_data_events.py"),
]


def _latest_ts_for_address(address: str):
    for d in DATA_DIRS:
        f = d / f"{address}.csv"
        if f.exists():
            try:
                df = pd.read_csv(f, usecols=["timestamp"])
                if len(df):
                    return int(float(df["timestamp"].iloc[-1]))
            except Exception:
                pass
    return None


now = int(time.time())
needs_fetch = False
missing = []
for name, addr in pools.items():
    ts = _latest_ts_for_address(addr)
    if ts is None:
        needs_fetch = True
        missing.append(name)
    else:
        age_min = (now - int(ts)) / 60.0
        if age_min > MIN_OLD:
            needs_fetch = True

if needs_fetch:
    for script in CANDIDATE_SCRIPTS:
        if script.exists():
            print(
                f"Data stale/missing ({', '.join(missing) if missing else 'stale'}) → running {script}..."
            )
            subprocess.run([sys.executable, str(script)], check=True)
            break
    else:
        print("Could not find fetch_data_events.py to refresh data.")
else:
    print("Data is up to date.")

In [ ]:
# Time window (None means full span)
# ts_null = 1758733775  # optional start cutoff to match analytics_all.ipynb
# ts_null = 1760180400 # ts post shakeout and vp = xcp_p/2
ts_null = 1

t_start = None
t_stop = None

# Load data
pools_data = load_pools_data(pools, decimals, ts_null=ts_null, data_dir="data_events")
list(pools_data.keys())
all_metrics = list(pools_data[list(pools_data.keys())[0]].keys())
print("All metrics:")
batch_print = 5
# print in batches of 3
for i in range(0, len(all_metrics), batch_print):
    print(f'  {", ".join(all_metrics[i:i+batch_print])}')

In [ ]:
# Pools to plot (dict, not list)
pools_to_plot = {
    "yb_cbBTC": False,
    "yb_wBTC": False,
    "yb_tBTC": True,
}
selected_pools = {k: pools[k] for k in ["yb_cbBTC", "yb_wBTC", "yb_tBTC"] if pools_to_plot[k]}
# Metric labels (legend-friendly names)
labels = {}

# Unified spec (mix of singles and groups)
spec = ["value_oracle", "value_pscale", "pscale_minus_oracle", "rel_pscale_minus_oracle"]

# Config with the requested 'plot' bool and behavior toggle
config = {
    "plot": True,
    "combine_grouped_on_one": True,  # default: per-pool subplots for grouped metrics
    "ncols": 2,  # auto columns (sqrt heuristic, capped at 3)
    "figsize_per": (4, 3),  # size per subplot (w, h)
    "sharex": True,
    "sharey": False,
}
t_start = time.time() - 14 * 24 * 3600
t_stop = time.time()
if config.get("plot", False):
    fig, axes = plot_unified_panel(
        selected_pools,
        pools_data,
        spec,
        config=config,
        labels=labels,
        pool_colors=pool_colors,
        pool_aliases=pool_aliases,
        t_start=t_start,
        t_stop=t_stop,
    )
    plt.show()
else:
    print('config["plot"] is False — not plotting. Adjust spec/config above and re-run.')

## Unified Panel: Spec and Config
- If an entry is a string metric (e.g., 'vp'), the subplot overlays all pools.
- If an entry is a list of metrics (e.g., ['xcp','xcp_half']), default behavior creates per‑pool subplots.
- Toggle `combine_grouped_on_one` to combine grouped metrics and pools into a single subplot.


In [ ]:
# Pools to plot (dict, not list)
pools_to_plot = {
    "yb_wBTC": 1,
    "yb_cbBTC": 1,
    "yb_tBTC": 1,
}

selected_pools = {k: pools[k] for k in ["yb_wBTC", "yb_cbBTC", "yb_tBTC"] if pools_to_plot[k]}
# Metric labels (legend-friendly names)
labels = {}

# Unified spec (mix of singles and groups)
spec = [
    "vp_xcp_half_gap",
    ["virtual_price", "xcp_profit_half"],
    "ps_oracle_diff",
    ["spot_mid", "price_oracle", "price_scale"],
    "crvusd_share",
    "balances_0",
    "balances_1",
    "value_spot_mid",
    "donations_value",
    ["donation_shares", "unlocked_shares", "available_shares"],
    "D",
    ["lp_price", "lp_price_scale"],
]
# spec = [
#     'vp_xcp_half_gap', 'D'
# ]

t_start = time.time() - 0.2 * 86_400

# Config with the requested 'plot' bool and behavior toggle
config = {
    "plot": True,
    "combine_grouped_on_one": False,  # default: per-pool subplots for grouped metrics
    "ncols": len(selected_pools.keys()) + 1,  # auto columns (sqrt heuristic, capped at 3)
    "figsize_per": (4, 3),  # size per subplot (w, h)
    "sharex": True,
    "sharey": False,
}
if config.get("plot", False):
    fig, axes = plot_unified_panel(
        selected_pools,
        pools_data,
        spec,
        config=config,
        labels=labels,
        pool_colors=pool_colors,
        pool_aliases=pool_aliases,
        t_start=t_start,
        t_stop=t_stop,
    )
    plt.show()
else:
    print('config["plot"] is False — not plotting. Adjust spec/config above and re-run.')